# DT401 Week 2 Section 2: Exploring Data Structures

**Module:** Data Fundamentals  
**Week:** 2 of 7  
**Section:** 2 of 4  
**Estimated Time:** 30–35 minutes

---

## Learning Objectives

By the end of this notebook, you will be able to:

- Identify common data structures (tables, lists, key-value pairs, trees) in a real dataset using Python output
- Explain the difference between mutable and immutable structures and why this matters for data integrity
- Recognise how Python data structures map to the learning classifications (table ↔ DataFrame, key-value pair ↔ dictionary, list/array ↔ Python list)
- Connect data structures to their SQL equivalents, preparing for Section 4 and Workshop 1

---

## Context Recap

In Section 1, you explored data *types* — integer, float, string, boolean — and classified them in a real weather dataset. In the Section 2 learning content, you learnt about data *structures* — how those individual values are organised into tables, lists, trees, and key-value pairs. This notebook brings those structures to life using Python, so you can see how they look and behave in practice. As with Section 1, your job is to **run** each cell, **read** the output, and **think** about what it means.

---

## A Note Before You Start

This notebook is about **exploring** data structures, not writing code. Every code cell in this notebook has already been written for you. Your job is to **run** each cell, **read** the output, and **think** about what it means.

You are not expected to write Python from scratch yet. That comes in DT402. Here, the code is a window into the data: you are looking through it, not building it.

Some tasks will ask you to change a single value in a code cell (for example, swapping a number or a key name) and re-run it. Others will ask you to fill in a table directly in a markdown cell. Instructions will always be explicit about exactly what to do.

---

## How to Use This Notebook

You will find three types of box throughout this notebook:

<div class="alert alert-block alert-warning">
<b>Reminder boxes:</b><br>
Important context about how this notebook connects to the other material you are studying or to your Module Challenge.
</div>

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>
Explanations of what a code cell did and what you should take from the output.
</div>

<div class="alert alert-block alert-success">
<b>Your turn:</b><br>
Tasks for you to complete. Some involve editing a single value in a code cell. Others ask you to fill in a table in a markdown cell. Instructions will always be explicit — you will never be asked to write code from scratch.

Tasks may include a hidden solution.
<details>
    <summary style="color:black;font-weight:bold">Click here to reveal the solution</summary>

Solutions appear here.

</details>
</div>

---

## Setup

Run the cell below to load the libraries this notebook uses and connect to a temporary database. You should see a confirmation message.

In [ ]:
import pandas as pd
import json
from IPython.display import display
%config SqlMagic.displaylimit = 0
%config SqlMagic.autopandas = True
%load_ext sql
%sql sqlite:///:memory:

# Load the same weather dataset used in Section 1
df = pd.read_csv('../open-meteo-51.49N0.16W23m.csv', skiprows=3, nrows=26328, low_memory=False,
                 converters={'is_day ()': lambda x: bool(float(x))})

print("Setup complete — pandas, json, and database ready.")
print(f"Weather dataset loaded: {len(df)} observations, {len(df.columns)} columns")

---

## Part 1: Tables and DataFrames — Structured Data

### What is a table?

The learning content introduced the **table** as the most familiar data structure: rows and columns, where each row is one record and each column is one attribute. You have already seen tables throughout your working life — spreadsheets, database reports, exported CSV files.

In Python, the equivalent structure is a **pandas [DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html)**. You encountered DataFrames in Section 1. This Part makes the connection between the learning concepts and the Python tool explicit, and shows how both relate to the database tables you will write in Section 4.

```mermaid
flowchart TD
    A[Table] --> B[Rows — individual records]
    A --> C[Columns — named attributes]
    A --> D[Headers — column names]
    B --> E[DataFrame: index]
    C --> F[DataFrame: columns and dtypes]
    D --> G[DataFrame: .columns property]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```

**What this diagram shows:** The components of a table — rows, columns, and headers — and how each one maps to the equivalent concept in a pandas DataFrame. The terminology changes between tools, but the structure is the same.

### Worked example 1: the weather dataset as a table

You have already loaded the weather dataset. Run the cell below to display a focused selection of columns and see the table structure clearly.

In [ ]:
# Display a focused selection of weather columns
cols_to_show = ['time', 'temperature_2m (°C)', 'rain (mm)', 'cloud_cover (%)', 'is_day ()']
display(df[cols_to_show].head(10))

# Show the dimensions: how many rows and columns
print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Column names: {df[cols_to_show].columns.tolist()}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

You are looking at a **table**. Each row is one hourly weather observation — a record. Each column is one attribute of that observation — temperature, rainfall, cloud cover, and so on.

The `.shape` tells you the table's dimensions: over 26,000 rows (hourly readings across three years) and multiple columns. The `.columns` property lists the column headers — just like the header row of a spreadsheet.

This is the table structure. The pandas DataFrame is Python's implementation of exactly this idea.
</div>

### Worked example 2: creating a table from scratch

You can also build a DataFrame directly — just as you would define a table in a database. Run the cell below to create a small table of NHS clinic appointments.

In [ ]:
# Create a small DataFrame representing NHS clinic appointments
clinic_data = pd.DataFrame({
    'date':           ['2026-01-06', '2026-01-07', '2026-01-08'],
    'patients':       [42,           38,            51],
    'avg_wait_mins':  [8.5,          11.2,          7.0],
    'clinic_open':    [True,         True,          True]
})

display(clinic_data)
print("\nColumn types:")
display(clinic_data.dtypes)

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Creating a DataFrame is like defining a database table. You choose the column names, and the data types follow from the values you provide: `patients` becomes an integer, `avg_wait_mins` becomes a float, `clinic_open` becomes a boolean.

This is the same `clinic_daily` table you saw in SQL form in the Section 1 notebook. The data is identical — only the tool is different.
</div>

### SQL preview: the same table in a database

Run the cell below to see how a database defines and queries the same clinic data.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS clinic_daily (
    date          TEXT,
    patients      INTEGER,
    avg_wait_mins DECIMAL(5,1),
    clinic_open   BOOLEAN
);

INSERT INTO clinic_daily VALUES
    ('2026-01-06', 42, 8.5,  TRUE),
    ('2026-01-07', 38, 11.2, TRUE),
    ('2026-01-08', 51, 7.0,  TRUE);

SELECT * FROM clinic_daily;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The DataFrame you just created and this SQL table contain exactly the same data. The DataFrame is Python's way of representing a table; SQL is the database's way. In Section 4 this week, you will write SQL to create and query tables like this one.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — guided modification:</b><br>

In Worked example 1 above, the cell displays `.head(10)` — the first 10 rows. Change `10` to `3` and re-run the cell to see fewer rows.

**Then try:** How many rows would `.head(20)` show? Change the number to `20` and run it to check.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
Think about tabular data in your organisation — a spreadsheet of customer records, a report exported from a database, a form with columns of responses. Each of these is a table. The DataFrame is how Python works with that same structure; the SQL table is how a database stores it.
</div>

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[What kind of data does pandas handle? — pandas documentation (pandas, 2026)](https://pandas.pydata.org/docs/getting_started/intro_tutorials/01_table_oriented.html) — A concise introduction to how pandas represents tabular data, with clear diagrams of rows, columns, and data types.

*Read time: 5 minutes*
</div>

<div class="alert alert-block alert-success">
<b>Your turn — classification task:</b><br>

Using what you observed in Worked example 1, complete the table below. Click on this cell to edit it, then fill in the empty columns.

For each column, decide: Is it a **row identifier** (used to locate a record), a **measurement** (a quantity that was observed or recorded), or a **category** (a label or flag)?
</div>

| Column | Sample value | Row identifier, measurement, or category? | data type (integer, float, string, boolean) |
|:-------|:------------|:------------------------------------------|:------------------------------------------------------|
| `time` | `2023-01-01T00:00` | | |
| `temperature_2m (°C)` | `11.2` | | |
| `rain (mm)` | `0.1` | | |
| `cloud_cover (%)` | `100.0` | | |
| `is_day ()` | `True` | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Column | Sample value | Row identifier, measurement, or category? | data type |
|:-------|:------------|:------------------------------------------|:--------------------|
| `time` | `2023-01-01T00:00` | Row identifier — locates each hourly observation | String (stored as text in this CSV) |
| `temperature_2m (°C)` | `11.2` | Measurement — a recorded quantity with decimal precision | Float |
| `rain (mm)` | `0.1` | Measurement — a recorded quantity with decimal precision | Float |
| `cloud_cover (%)` | `100.0` | Measurement — a percentage, though stored as a decimal | Float |
| `is_day ()` | `True` | Category — a binary flag: daytime or night | Boolean |

</details>

---

## Part 2: Lists and Arrays — Ordered Collections

### What is a list?

The learning content introduced two related structures: **arrays** (same type, fixed size, accessed by position) and **lists** (mixed types, flexible size, mutable). In Python, the primary structure for ordered collections is the **list** — and understanding lists means understanding the most important concept in this Part: **mutability**.

A **mutable** structure can be changed after it is created — you can add items, remove items, or reorder them. An **immutable** structure cannot be changed once it is created. Python has a structure for each: **lists** are mutable; **tuples** are immutable.

| **List** | **Tuple** | **Array** |
|:----:|:-----:|:-----:|
| Mutable | Immutable | Fixed size |
| Mixed data types | Mixed data types | Same data type |
| Indexed by position | Indexed by position | Indexed by position |
| One dimensional | One dimensional | Can be multi-dimensional |
| Can contain other structures | Can contain other structures | Cannot contain other structures |

**What this diagram shows:** Both arrays and lists are ordered sequences accessed by position. The key difference is flexibility: arrays are rigid (same type, fixed size), while lists can grow, shrink, and hold mixed types. Python's tuple sits closest to the array concept — immutable and position-based — while the Python list is more flexible than either.

### Worked example 1: extracting a list from the weather data

Run the cell below to extract the column names from the weather DataFrame as a Python list.

In [ ]:
# Extract column names as a list
column_list = df.columns.tolist()

print(f"Number of columns: {len(column_list)}")
print(f"First column: {column_list[0]}")
print(f"Last column:  {column_list[-1]}")
print(f"\nAll columns: {column_list}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

A Python list is an ordered collection. Items are accessed by position — `column_list[0]` is the first item, `column_list[-1]` is the last (negative indexing counts from the end). 

The column names here are all strings — the same type. This makes this particular list behave more like an array: uniform, ordered, accessed by position.
</div>

### Worked example 2: lists are mutable

Run the cell below to see how a patient queue changes as patients arrive and are seen.

In [ ]:
# Patient queue — starts with three patients
queue = ["P001", "P002", "P003"]
print(f"Starting queue: {queue}")

# A new patient arrives
queue.append("P004")
print(f"After P004 arrives: {queue}")

# The first patient is seen and leaves the queue
seen = queue.pop(0)
print(f"Patient seen: {seen}")
print(f"Remaining queue: {queue}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Lists can change after creation — you can add, remove, and reorder items. This is called **mutability**. We noted that lists are mutable while arrays are immutable (fixed). In practice, Python lists are the go-to structure for data that changes over time: queues, task lists, growing collections of results.

`.append()` adds an item to the end. `.pop(0)` removes and returns the item at position 0 — the front of the queue.
</div>

### Worked example 3: tuples are immutable

A tuple looks like a list but uses parentheses instead of square brackets — and it cannot be changed. Run the cell below to see what happens when you try to modify one.

In [ ]:
# Database connection settings — these should never change at runtime
config = ("localhost", 5432, "weather_db")

print(f"Host:     {config[0]}")
print(f"Port:     {config[1]}")
print(f"Database: {config[2]}")

# Now try to change the host — this will cause an error
try:
    config[0] = "newhost"
except TypeError as e:
    print(f"\nAttempted to modify a tuple and got: {e}")
    print("Tuples cannot be changed after creation — this is intentional.")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Tuples look like lists but cannot be changed. Python raises a `TypeError` immediately when you attempt to modify one. Use tuples when data must be protected from accidental modification — coordinates, configuration settings, historical records that must not be altered.
</div>

### SQL preview: a column as an ordered list of values

Run the cell below to see how a SQL query returns a single column — the database equivalent of a list of values of one type.

In [ ]:
%%sql
SELECT date, patients
FROM clinic_daily
ORDER BY date;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

A single column in a SQL table — here, `patients` — is like an array: all values are the same type (INTEGER), ordered, and accessed by position (row number). SQL enforces this consistency. Python lists are more flexible, but when data enters a database column, it must conform to the column's declared type.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — guided modification:</b><br>

In Worked example 2 above, the patient queue starts with `["P001", "P002", "P003"]`.

Change `"P001"` to include your own initials instead — for example, if your initials are AB, change it to `"AB01"`. Re-run the cell and observe that the output changes to reflect your new value.

This is all you need to change — the rest of the code handles the queue operations automatically.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
Patient queues, task lists, and product catalogues that grow and shrink are lists. Configuration settings, geographic coordinates, and version numbers that must not change are tuples. Think about which data in your role changes frequently (list) and which must stay fixed (tuple).
</div>

<div class="alert alert-block alert-success">
<b>Your turn — classification task:</b><br>

For each data example below, classify it as best contained in a **List (mutable — changes expected)** or a **Tuple (immutable — should not change)**, and write a brief reason. Edit this cell to fill in the empty columns.
</div>

| Data example | List or Tuple? | Reason |
|:-------------|:---------------|:------------------------------------------------|
| Products currently in a customer's online shopping cart | | |
| GPS coordinates of a warehouse in Birmingham | | |
| Employee IDs currently logged into a system | | |
| Historical dates on which a board meeting was held | | |
| Support tickets waiting to be assigned | | |
| The version number of a software release (e.g., 3, 1, 2) | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Data example | List or Tuple? | Reason |
|:-------------|:---------------|:-------|
| Products currently in a customer's online shopping cart | List | Items are added and removed as the customer shops |
| GPS coordinates of a warehouse in Birmingham | Tuple | The warehouse location does not change; protecting it from accidental modification prevents data errors |
| Employee IDs currently logged into a system | List | The set of logged-in employees changes constantly |
| Historical dates on which a board meeting was held | Tuple | Historical records should not be changeable; immutability protects their integrity |
| Support tickets waiting to be assigned | List | Tickets are added when raised and removed when assigned |
| The version number of a software release (e.g., 3, 1, 2) | Tuple | A released version number is fixed; it should not be modified after release |

</details>

---

## Part 3: Dictionaries — Key-Value Pairs

### What is a key-value pair?

We introduced **key-value pairs** as a structure that maps unique identifiers (keys) to associated data (values) — like a real dictionary maps words to definitions, or a database record maps field names to field values.

In Python, the key-value pair structure is the **dictionary**. Dictionaries are one of the most important structures in Python data work: configuration files, API responses, database records, and JSON files all use this pattern.

```mermaid
flowchart TD
    A[Dictionary] --> B[Keys — unique identifiers]
    A --> C[Values — any type]
    B --> D[String: 'temperature_2m']
    B --> E[String: 'rain']
    B --> F[String: 'is_day']
    C --> G[Float: 11.2]
    C --> H[Float: 0.1]
    C --> I[Boolean: False]

    style A fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style B fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style C fill:#ffffff,stroke:#000000,color:#000000,stroke-width:2px
    style D fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style E fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style F fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style G fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style H fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
    style I fill:#ffffff,stroke:#000000,color:#000000,stroke-width:1px
```

**What this diagram shows:** A dictionary maps unique keys to values. Each key is a string that names a field; each value holds the data for that field. The values can be of any type — float, boolean, string, or even another dictionary.

### Worked example 1: a single weather observation as a dictionary

Run the cell below to extract one row from the weather DataFrame and represent it as a dictionary.

In [ ]:
# Extract the first observation and convert to a dictionary
cols_for_dict = ['time', 'temperature_2m (°C)', 'rain (mm)', 'is_day ()']
observation = df.iloc[0][cols_for_dict].to_dict()

print("One weather observation as a dictionary:")
print(observation)

# Access individual values by key
print(f"\nTemperature: {observation['temperature_2m (°C)']} °C")
print(f"Is daytime:  {observation['is_day ()']}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Each row of the weather DataFrame can be represented as a dictionary — the column names become keys, and the cell values become values.

Notice how you access values: by key name (`observation['temperature_2m (°C)']`), not by position. This is more readable than `observation[1]` — you do not need to remember which index corresponds to which field.
</div>

### Worked example 2: building a structured record

Run the cell below to create a dictionary representing an NHS patient record.

In [ ]:
# An NHS patient record as a dictionary
patient = {
    "id":        "PAT-2026-0471",
    "name":      "Amara Okafor",
    "age":       34,
    "consented": True,
    "ward":      "Cardiology"
}

print(f"Patient:   {patient['name']}")
print(f"ID:        {patient['id']}")
print(f"Ward:      {patient['ward']}")
print(f"Consented: {patient['consented']}")

# Add a new field
patient["bed_number"] = 14
print(f"\nAfter adding bed number: {patient}")

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

Dictionaries store structured records naturally. Each field has a descriptive key — you look up data by name, not by position. This is far more intuitive than accessing `patient[2]` in a list (which field is index 2?).

Dictionaries are also mutable: you added `bed_number` after the record was created, and the dictionary updated immediately. This is different from a tuple — records in a database can have new fields added, but their structure is governed by a schema.
</div>

### Worked example 3: nested dictionaries and JSON

We showed JSON as a data interchange format built on key-value pairs. Run the cell below to see the connection between a Python dictionary and JSON.

In [ ]:
# A product catalogue entry with nested specifications
product = {
    "PID101": {
        "name": "Bluetooth Speaker",
        "price": 19.99,
        "specs": {
            "battery_hours": 20,
            "waterproof": True
        }
    }
}

# Display as a Python dictionary
print("As a Python dictionary:")
display(product)

# Convert to JSON (the format used by APIs and web services)
print("\nAs JSON (for sending to an API or saving to a file):")
print(json.dumps(product, indent=2))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

You previously learnt about how JSON can be used as a data interchange format built on key-value pairs. Python dictionaries and JSON are almost identical in structure — and `json.dumps()` converts one to the other instantly.

The nested structure here also shows the beginning of a **tree**: the product has a `specs` key whose value is itself a dictionary. This is how the tree structure we previously explored appears in practice — nested key-value pairs, with parent records containing child records.

When you work with web APIs — weather services, financial data feeds, customer databases — the data arrives as JSON, which Python reads as dictionaries.
</div>

### SQL preview: a dictionary as one row of a table

Run the cell below to see the same patient record stored in a SQL table.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS patients (
    id         VARCHAR(20),
    name       VARCHAR(100),
    age        INTEGER,
    consented  BOOLEAN,
    ward       VARCHAR(50)
);

INSERT INTO patients VALUES
    ('PAT-2026-0471', 'Amara Okafor', 34, TRUE, 'Cardiology');

SELECT * FROM patients;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

A dictionary holds one record with named fields. A SQL table holds many records with the same fields defined as columns. The dictionary is like one row of the table — the keys correspond to column names, and the values correspond to cell contents.

In Section 4, you will create tables like this and query them with SELECT.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — guided modification:</b><br>

In Worked example 2, the patient's ward is `"Cardiology"`. Change this to a department relevant to your own workplace — or invent one — and re-run the cell.

Notice that only the value for the `"ward"` key changes; the structure of the dictionary remains the same.
</div>

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
Configuration files, API responses, customer profiles, product catalogues — all use key-value structures. JSON files that configure your software, API calls that retrieve data from external services, and the records in your organisation's databases all follow this pattern.
</div>

<div class="alert alert-block alert-info">
<b>Reinforcement Resource</b><br>

[JSON Introduction — W3Schools (W3Schools, 2026)](https://www.w3schools.com/js/js_json_intro.asp) — A concise overview of JSON syntax and structure, showing how key-value pairs are formatted for data interchange.

*Read time: 5 minutes*
</div>

<div class="alert alert-block alert-success">
<b>Your turn — classification task:</b><br>

For each data scenario below, identify which theoretical structure you have studied it most closely resembles. Edit this cell to fill in the empty column.

Choose from: 

- **Table**
- **Array / List**
- **Key-Value Pair / Dictionary**
- **Tree**
</div>

| Data scenario | Useful data structure |
|:--------------|:--------------------|
| A JSON response from a weather API containing temperature, rainfall, and wind speed with named fields | |
| A spreadsheet of 500 employee records, each with the same columns | |
| A configuration file with server settings: host, port, timeout, and retry limit | |
| A product category hierarchy: Electronics → Audio → Headphones → Over-ear | |
| A queue of support tickets waiting to be assigned, stored in order of arrival | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Data scenario | Useful data structure |
|:--------------|:--------------------|
| A JSON response from a weather API containing temperature, rainfall, and wind speed with named fields | Key-Value Pair / Dictionary — named fields map directly to keys and values |
| A spreadsheet of 500 employee records, each with the same columns | Table — rows are records, columns are attributes, structure is uniform |
| A configuration file with server settings: host, port, timeout, and retry limit | Key-Value Pair / Dictionary — each setting is named and has a single value |
| A product category hierarchy: Electronics → Audio → Headphones → Over-ear | Tree — parent-child relationships branch progressively from broad to specific |
| A queue of support tickets waiting to be assigned, stored in order of arrival | Array / List — ordered by position (arrival time), items join at the end and leave from the front |

</details>

---

## Part 4: Connecting Structures to Your Workplace and to SQL

### Bringing it together

The three previous Parts each explored one data structure in isolation. In real systems, structures are combined. This final Part shows the most common combination — a list of dictionaries — and then maps all four structures to their SQL equivalents, giving you a complete bridge to the SQL work in Section 4.

### Worked example 1: a list of dictionaries

The most common data pattern in Python data work is a list of dictionaries — multiple records, each structured the same way. Run the cell below to see this pattern with weather observations.

In [ ]:
# Three weather observations — each a dictionary, all collected in a list
observations = [
    {"time": "2023-01-01T00:00", "temp": 11.2, "rain": 0.1, "is_day": False},
    {"time": "2023-01-01T01:00", "temp": 11.9, "rain": 0.4, "is_day": False},
    {"time": "2023-01-01T02:00", "temp": 11.0, "rain": 0.0, "is_day": False}
]

# Convert to a DataFrame — a list of dicts becomes a table
print("As a DataFrame (table):")
display(pd.DataFrame(observations))

# Convert to JSON — the interchange format
print("\nAs JSON (for sending to an API or saving to a file):")
print(json.dumps(observations, indent=2))

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

A list of dictionaries is the bridge between all the structures you have seen. Each dictionary is one record — like one row. The list collects them in order — like a table's rows. Convert to a DataFrame and you have a table. Convert to JSON and you have an interchange format.

This is the fundamental pattern behind most data systems: records (dictionaries) collected into sequences (lists), stored in databases (tables), and transmitted between systems (JSON).
</div>

### Worked example 2: the same data in SQL

Run the cell below to store and query the same three observations in a SQL table.

In [ ]:
%%sql
CREATE TABLE IF NOT EXISTS weather_obs (
    time   TEXT,
    temp   DECIMAL(4,1),
    rain   DECIMAL(4,1),
    is_day BOOLEAN
);

INSERT INTO weather_obs VALUES
    ('2023-01-01T00:00', 11.2, 0.1, FALSE),
    ('2023-01-01T01:00', 11.9, 0.4, FALSE),
    ('2023-01-01T02:00', 11.0, 0.0, FALSE);

-- Filter for observations where it was raining
SELECT * FROM weather_obs WHERE rain > 0;

<div class="alert alert-block alert-info">
<b>What just happened?</b><br>

The same three records, now in SQL. The `WHERE rain > 0` filter is the SQL equivalent of a pandas filter — it returns only the rows where a condition is met.

In Section 4, you will write queries like this yourself. The structural knowledge from this notebook is the foundation you will build on.
</div>

### The complete structure map

Every data structure you learnt about in the theory has a Python equivalent and a SQL equivalent. The diagram below shows the complete mapping.

| Theoretical structure | Python equivalent | SQL equivalent |
|:---------------------|:------------------|:---------------|
| Table | DataFrame | Table |
| List / Array | List / Array | Column (single type) |
| Key-Value Pair | Dictionary | Row (named fields) |
| Tree | Nested dictionaries / JSON | Hierarchical tables with foreign keys |

**What this diagram shows:** Every data structure from the learning materials has a Python equivalent and a SQL equivalent. The concepts stay the same; only the tool and syntax change. Understanding this mapping means you can move between Python, SQL, and everyday data tools — you are working with the same underlying ideas throughout.

<div class="alert alert-block alert-warning">
<b>Workplace connection:</b><br>
When your organisation stores data in a database, queries it with SQL, exports it to CSV, and sends it via a JSON API — it is moving between these equivalent representations. The data does not change; only the format does. This is why understanding data structures, rather than just one tool, is so valuable.
</div>

<div class="alert alert-block alert-success">
<b>Your turn — comprehensive classification task:</b><br>

Complete the table below for eight real-world data scenarios across different sectors. For each one, identify the theoretical structure name, its Python equivalent, its SQL equivalent, and whether the data is typically mutable or immutable.

Edit this cell directly to fill in the empty columns.
</div>

| Data scenario | Theoretical structure | Python equivalent | SQL equivalent | Mutable or Immutable? |
|:--------------|:--------------------|:------------------|:---------------|:----------------------|
| NHS patient list — rows of patient records with consistent columns | | | | |
| Tesco product inventory — each product has a SKU, name, price, and stock count | | | | |
| Barclays transaction log — timestamped entries in arrival order | | | | |
| Council service request queue — tickets awaiting assignment | | | | |
| British Red Cross donor profile — named fields for one supporter | | | | |
| University module enrolment — spreadsheet of students and their enrolled modules | | | | |
| Arm chip specification — nested technical properties (architecture → cores → clock speed) | | | | |
| DeepMind model configuration — named settings: learning rate, batch size, epochs | | | | |

<details>
    <summary style="font-weight:bold">Click here to reveal the solution</summary>

| Data scenario | Theoretical structure | Python equivalent | SQL equivalent | Mutable or Immutable? |
|:--------------|:--------------------|:------------------|:---------------|:----------------------|
| NHS patient list | Table | DataFrame | TABLE | Mutable — records are added, updated, and discharged |
| Tesco product inventory | Key-Value Pair / Dictionary | dictionary | Row (one record in a TABLE) | Mutable — prices and stock counts change |
| Barclays transaction log | Array / List | list | Column (ordered sequence of values) | Immutable — historical transactions must not be altered |
| Council service request queue | Array / List | list | Column or TABLE (ordered records) | Mutable — tickets are added and removed |
| British Red Cross donor profile | Key-Value Pair / Dictionary | dictionary | Row (one record in a TABLE) | Mutable — contact details and preferences may change |
| University module enrolment | Table | DataFrame | TABLE | Mutable — enrolments are added and withdrawn |
| Arm chip specification | Tree | Nested dictionary | Related tables with foreign keys | Immutable — once a chip is released, its spec is fixed |
| DeepMind model configuration | Key-Value Pair / Dictionary | dictionary | Row or separate config TABLE | Mutable during development; immutable once a model is deployed |

**Note:** Some scenarios have reasonable arguments for either mutable or immutable — for example, a donor profile might be considered immutable for historical records but mutable for current contact details. The key skill is justifying your reasoning, not finding the single correct answer.

</details>

<div class="alert alert-block alert-warning">
<b>Module Challenge connection:</b><br>
Classifying data structures and explaining how your organisation's systems store and organise data is a key component of Module Challenge Part A. Your completed classification tables and reflections from this notebook provide portfolio evidence of this skill. The ability to identify a structure, name its Python equivalent, and connect it to its SQL representation is exactly what the portfolio is designed to demonstrate.
</div>

---

## Reflection

Reflect on the following question and record your thoughts in the cell below. Ground your answer in an example from your workplace, using **approximately 150 words** in total.

Your answer here forms part of your Module Challenge Part A portfolio and is assessed against **MLO2:** *Critically reflect upon data management systems and their role in digital and technology solutions*.

Where you draw on external sources, cite them in Harvard style. This does not count towards your word count.

Once you commit your notebook, your Coach will be able to provide feedback on your response. You can develop your answer further before your final submission.

<div class="alert alert-block alert-info">

1. Using this week's distinction between mutable and immutable data, identify one situation in your role where data *should* be protected from change, where accidental modification would cause a problem. Then identify one where it *must* be changeable, where the data needs to be updated to reflect changes and the old value is no longer valid.Does your workplace's current handling of this distinction get the balance right? Justify your view.

</div>

*Write your responses here.*

1. 

References (if applicable):

---

## Summary

**You have completed Week 2 Section 2.** You can now:

- Represented the weather dataset as a pandas DataFrame and built a new DataFrame from scratch, then declared the equivalent SQL table
- Distinguished mutable lists from immutable tuples using a patient queue and a database configuration tuple, observing a `TypeError` when attempting to modify the tuple
- Represented weather observations and an NHS patient record as Python dictionaries, then converted a nested dictionary to JSON with `json.dumps()`
- Combined a list of dictionaries into a DataFrame and a SQL table, then completed the full structure map linking each theoretical structure to its Python and SQL equivalent

**Key takeaways:**

- Mutability determines the right structure for data that changes (lists, dictionaries) versus data that must stay fixed (tuples)
- Every theoretical structure has a Python equivalent and a SQL equivalent: table to DataFrame to SQL table, list to Python list to column, key-value pair to dictionary to row, tree to nested dictionary to related tables with foreign keys

**What comes next:**

- **Section 3** explores data formats: CSV, JSON, XML, YAML, and Markdown, building on the structures you have seen here. You have already seen how a Python dictionary converts to JSON; Section 3 goes deeper into how formats are chosen and used.
- **Section 4** introduces SQL in depth. You will write your own `CREATE TABLE`, `INSERT`, and `SELECT` queries using the structural knowledge from this notebook, using the mapping diagram in Part 4 as your bridge to that work.

**Action items:**

1. Save this notebook (Ctrl+S or Cmd+S)
2. Commit and push your work to your GitHub repository with the commit message `"Complete Week 2 Section 2 data structures exploration"`
3. Make sure the repository shows your latest commit and that all files are up to date
4. Share a link to your repository with your Coach in the DT401.2 component on Aptem
5. Make sure your Coach has access to your repository so they can view your work and provide feedback
6. Open `DT401_Week_2_Section_3.ipynb` when ready to continue
